# Calling Azure OpenAI Models via APIM

This guide shows how to switch from calling Azure OpenAI directly to routing through an **APIM endpoint**.

## Summary of Changes

| Item | Before (Direct Call) | After (Via APIM) |
|------|---------------------|-----------------|
| Endpoint | `https://{resource}.openai.azure.com` | `https://{apim-name}.azure-api.net` |
| API Key | Foundry Project API Key | **APIM Subscription Key** |
| Code Change | - | Only change `azure_endpoint` and `api_key` |

## Prerequisites

1. Set the following values in your `.env` file:
   - `APIM_ENDPOINT`: APIM gateway URL
   - `APIM_SUBSCRIPTION_KEY`: APIM Subscription Key shared by your team
   - `DEPLOYMENT_NAME`: Model deployment name (e.g., gpt-4o)
   - `API_VERSION`: API version (e.g., 2024-12-01-preview)

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()

apim_endpoint = os.getenv("APIM_ENDPOINT")
apim_key = os.getenv("APIM_SUBSCRIPTION_KEY")
deployment_name = os.getenv("DEPLOYMENT_NAME", "gpt-4o")
api_version = os.getenv("API_VERSION", "2024-12-01-preview")

print(f"APIM Endpoint: {apim_endpoint}")
print(f"Deployment: {deployment_name}")

APIM Endpoint: https://apim-foundry-gw-jihys.azure-api.net
Deployment: gpt-4o


## Direct Call vs. APIM-Routed Call

Previous code calling Azure OpenAI directly:

```python
# Previous approach (direct call)
client = AzureOpenAI(
    azure_endpoint="https://my-resource.openai.azure.com",
    api_key="foundry-project-api-key",
    api_version="2024-12-01-preview",
)
```

Simply change `azure_endpoint` and `api_key` to your APIM values.
The rest of the code remains **exactly the same**.

In [ ]:
# Call via APIM — only endpoint and key are changed
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=apim_endpoint,
    api_key=apim_key,
    api_version=api_version,
)

response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is Azure API Management? Explain in one sentence."},
    ],
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens} (prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens})")

Azure API Management는 API를 보호, 게시, 모니터링 및 관리할 수 있도록 지원하는 Microsoft Azure의 완전 관리형 서비스입니다.

사용 토큰: 64 (prompt: 32, completion: 32)


## Token Usage Tracking

Token usage from the above call is **automatically recorded** via App Insights Telemetry.
You can view per-project and per-model usage and estimated costs in the **Cost Dashboard** on the Azure Portal.

> 💡 The dashboard is auto-provisioned during Terraform deployment. No additional setup is required.

## Per-User Usage Tracking

When you call the API with a Personal Key, per-user usage is **tracked automatically**.

The APIM Product Policy extracts token usage from each response and logs it to App Insights with the following dimensions:

| Dimension | Description |
|-----------|-------------|
| `subscriber` | User email or ID (auto-identified from Personal Key) |
| `project-name` | Project the user belongs to (Product name) |
| `model` | Model name used |
| `total-tokens` | Total token count |
| `prompt-tokens` | Input token count |
| `completion-tokens` | Output token count |

> **No custom headers or additional configuration required.** Using a Personal Key is all that's needed for user identification.

Usage can be viewed at Azure Portal → App Insights → Workbooks.